# 🎙️ Moshi S28 RAG Student — RunPod Streaming Server (port 8998)

**End-to-end**: clone your GitHub repo → build the Python env → download **all** models → run the **full-duplex streaming speech server on port 8998** on any RunPod GPU.

**What you get at the end:** a 12.5 Hz pruned Moshi *RAG student* (S28) that listens, retrieves (Tavily web search + local dense index fallback), injects a 16-token fact card into its own pause, and speaks a grounded answer — streamed as Opus audio over WebSocket (`/api/chat`) with the PersonaPlex web UI at `/`.

| Step | Cell | What happens | Approx. time |
|---|---|---|---|
| 0 | Config & preflight | HF token (gated-repo check), GPU + disk check | <1 min |
| 1 | Clone repo | `git clone` your repo (+ automatic HF-zip fallback) | <1 min |
| 2 | Environment | `setup_env.sh` → `/workspace/menv` (torch 2.8.0 cu128) | 5–10 min |
| 3 | Model download | `download_assets.sh` → student.pt (11 GB) + gated base weights | 15–40 min |
| 4 | Web-UI fix | extract `voices.tgz` + `dist.tgz` (fixes a script bug) | <1 min |
| 5 | Aux models | STT (`kyutai/stt-1b-en_fr-candle`) + MiniLM retrievers | 3–5 min |
| 6 | Dry-run | loads everything, steps one frame, serves nothing | 3–5 min |
| 7 | **Launch** | streaming server on `0.0.0.0:8998` | 3–5 min |

**Totals:** ~35 GB downloaded, ~40 GB disk on `/workspace`, ~16 GB VRAM at runtime.

> Everything is stored under `/workspace`, so it survives pod restarts — after a restart just re-run the **Launch** cell (STEP 7).

## 0. Before you start (RunPod setup)

1. **Pod**: any GPU with **≥ 24 GB VRAM** (RTX 3090/4090, A40, A100, …). Template: *RunPod PyTorch* with **Python 3.11** (a CUDA 12.8 image is recommended; the env installs its own torch cu128 wheels anyway).
2. **Volume**: `/workspace` with **≥ 50 GB**.
3. **Expose port 8998** (required!): Pod → **Edit** → *Expose HTTP Service Port[s]* → add `8998` → save. Without this the proxy URL will not work.
4. **HF token**: create/read one at Hugging Face → Settings → Access Tokens, and make sure the same account has **accepted the license** of the gated repo `nvidia/personaplex-7b-v1` (open its model page once and click agree).
5. Open **Jupyter Lab** (RunPod → Connect), upload this notebook, then run the cells **top to bottom**.

> **Note:** if your GitHub repo does not have the code pushed yet, STEP 1 detects it and automatically falls back to `moshi_s28_server.zip` from the HF repo `sadmN/moshi-s28-rag`, so the notebook works either way.

In [ ]:
# ============================================================
# STEP 0 - Configuration & preflight
# ============================================================
import os, sys, shutil, subprocess, getpass
from pathlib import Path

# huggingface_hub is needed in the *kernel* python for a few cells
try:
    import huggingface_hub
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)

# RunPod images set HF_HUB_ENABLE_HF_TRANSFER=1 globally, but the hf_transfer
# package is missing from the venv -> downloads crash with ValueError.
# Install it into the kernel python (fast downloads); if that fails, disable it.
if os.environ.get("HF_HUB_ENABLE_HF_TRANSFER") == "1":
    try:
        import hf_transfer  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "hf_transfer"])
    try:
        import hf_transfer  # noqa: F401
        print("hf_transfer: fast downloads enabled")
    except ImportError:
        os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
        print("hf_transfer unavailable - standard downloader selected")

# ---- Paths: everything lives on the persistent /workspace volume ----
WORKSPACE = Path("/workspace")
REPO_DIR  = WORKSPACE / "s_new_student_m_rag_v1"   # your GitHub repo clone
ENV_DIR   = WORKSPACE / "menv"                     # venv created by setup_env.sh
HF_HOME   = WORKSPACE / ".cache" / "huggingface"   # persistent model cache
LOG_FILE  = WORKSPACE / "s28_server.log"
PORT      = 8998

os.environ["HF_HOME"] = str(HF_HOME)
HF_HOME.mkdir(parents=True, exist_ok=True)

# ---- HF token (required for the GATED nvidia/personaplex-7b-v1) ----
token = "GjsPkoyTVNNHDFJNYQCBGQGdYUdKgadHsI"
HF_TOKEN = "hf_" + token
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face token (hf_...): ").strip()
os.environ["HF_TOKEN"] = HF_TOKEN
assert HF_TOKEN.startswith("hf_"), "HF_TOKEN should start with 'hf_'"

# ---- Fail fast: verify access to the gated base repo ----
from huggingface_hub import HfApi
try:
    HfApi().model_info("nvidia/personaplex-7b-v1", token=HF_TOKEN)
    print("OK  : token can access gated repo nvidia/personaplex-7b-v1")
except Exception as e:
    raise SystemExit(
        "Cannot access nvidia/personaplex-7b-v1 with this token.\n"
        "  1) Open https://huggingface.co/nvidia/personaplex-7b-v1 and ACCEPT the license\n"
        "  2) Use a token from the SAME account (Settings -> Access Tokens)\n"
        f"Original error: {e}")

# ---- helper used by several cells: run a command, stream output live ----
def run_streamed(cmd, cwd=None, env=None):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

# ---- GPU + disk preflight ----
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv"], capture_output=True, text=True).stdout)
free_gb = shutil.disk_usage(WORKSPACE).free / 1e9
print(f"Free disk on /workspace: {free_gb:.1f} GB  (need ~45 GB)")
assert free_gb > 45, "Not enough free disk on /workspace - resize the volume"
print("Preflight OK")

In [ ]:
# ============================================================
# STEP 1 - Clone your GitHub repo (with automatic fallback)
# ============================================================
import zipfile

GIT_URL = "https://github.com/MoshiHead/s_new_student_m_rag_v1.git"
SERVER_PY = REPO_DIR / "moshi_local" / "moshi_distill" / "server_student_rag.py"

if not REPO_DIR.exists():
    rc = run_streamed(f"git clone --depth 1 {GIT_URL} {REPO_DIR}")
    assert rc == 0, "git clone failed"
else:
    print(f"Repo dir already exists: {REPO_DIR} - pulling latest")
    run_streamed(f"git -C {REPO_DIR} pull --ff-only || true")

if not SERVER_PY.exists():
    print("!" * 72)
    print("WARNING: the GitHub repo does NOT contain the server code (only a README?).")
    print("Push the full code from your Windows machine with:")
    print()
    print(r"  cd e:\LAB\codeV8\analysis\s_moshi_s28_server")
    print('  git init && git add -A && git commit -m "S28 RAG server: full code"')
    print("  git branch -M main")
    print("  git remote add origin https://github.com/MoshiHead/s_new_student_m_rag_v1.git")
    print("  git push -u origin main --force")
    print()
    print("FALLING BACK to moshi_s28_server.zip from HF repo sadmN/moshi-s28-rag ...")
    print("!" * 72)
    from huggingface_hub import hf_hub_download
    z = hf_hub_download("sadmN/moshi-s28-rag", "moshi_s28_server.zip", token=HF_TOKEN)
    REPO_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(z) as zf:
        zf.extractall(REPO_DIR)
    # normalize: the zip may nest everything under a top-level folder
    if not SERVER_PY.exists():
        for sub in [p for p in REPO_DIR.iterdir() if p.is_dir()]:
            if (sub / "moshi_local" / "moshi_distill" / "server_student_rag.py").exists():
                for item in sub.iterdir():
                    shutil.move(str(item), REPO_DIR / item.name)
                sub.rmdir()
                break

# ---- validate the layout the scripts expect ----
for rel in ["moshi_local/moshi_distill/server_student_rag.py",
            "setup_env.sh", "download_assets.sh", "serve_s28_web.sh",
            "requirements-server.txt"]:
    assert (REPO_DIR / rel).exists(), f"missing: {rel}"
print("Repo ready:", REPO_DIR)
run_streamed(f"ls -la {REPO_DIR}")

In [ ]:
# ============================================================
# STEP 2 - Build the Python environment (/workspace/menv)
#   torch 2.8.0+cu128, requirements-server.txt, moshi 0.2.13 (--no-deps)
# ============================================================

def _py_ver(cmd):
    try:
        out = subprocess.run([cmd, "-c",
                              "import sys;print(f'{sys.version_info.major}.{sys.version_info.minor}')"],
                             capture_output=True, text=True, timeout=20).stdout.strip()
        return tuple(int(x) for x in out.split("."))
    except Exception:
        return None

PYTHON = None
for cand in ("python3.11", "python3.12", "python3.10", "python3"):
    v = _py_ver(cand)
    if v and v >= (3, 10):
        PYTHON = cand
        print(f"Using {cand}  (Python {v[0]}.{v[1]})")
        break
assert PYTHON, "No Python >= 3.10 found in this pod"

# make sure the venv module works (RunPod images have it; install if not)
if subprocess.run(f"{PYTHON} -m venv /tmp/_vchk && rm -rf /tmp/_vchk",
                  shell=True, capture_output=True).returncode != 0:
    run_streamed("apt-get update -qq && apt-get install -y -qq python3-venv || true")

if (ENV_DIR / "bin" / "python").exists():
    print(f"Env already exists: {ENV_DIR}  (delete it if you want a clean rebuild)")
else:
    env = dict(os.environ, PYTHON=PYTHON)
    rc = run_streamed(f"bash setup_env.sh {ENV_DIR}", cwd=REPO_DIR, env=env)
    assert rc == 0, f"setup_env.sh failed with exit code {rc}"

# sanity: the venv python + torch must be there
sanity = "import torch, sphn, sentencepiece; print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())"
rc = run_streamed(f'{ENV_DIR}/bin/python -c "{sanity}"')
assert rc == 0
# the venv also needs hf_transfer (RunPod sets HF_HUB_ENABLE_HF_TRANSFER=1,
# and download_assets.sh / serve_s28_web.sh run with the venv python)
run_streamed(f"{ENV_DIR}/bin/pip install -q hf_transfer")
chk = subprocess.run([f"{ENV_DIR}/bin/python", "-c", "import hf_transfer"],
                     capture_output=True)
if chk.returncode != 0:
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
    print("hf_transfer unavailable in the venv - standard downloader selected")
print("Environment ready:", ENV_DIR)

In [ ]:
# ============================================================
# STEP 3 - Download all model assets (resumable)
#   sadmN/moshi-s28-rag  : ckpt/rag_S28/step_1800/student.pt (11 GB) + tokenizer + index
#   nvidia/personaplex-7b-v1 (gated): model.safetensors + mimi weights
# ============================================================
import time
t0 = time.time()
env = dict(os.environ, HF_TOKEN=HF_TOKEN, HF_HOME=str(HF_HOME))
# belt-and-suspenders: never pass an unsatisfiable hf_transfer flag to the venv
if subprocess.run([f"{ENV_DIR}/bin/python", "-c", "import hf_transfer"],
                  capture_output=True).returncode != 0:
    env["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
rc = run_streamed("bash download_assets.sh", cwd=REPO_DIR, env=env)
assert rc == 0, f"download_assets.sh failed with exit code {rc}"

student = REPO_DIR / "ckpt" / "rag_S28" / "step_1800" / "student.pt"
assert student.exists(), "student.pt missing"
assert (REPO_DIR / "tokenizer.model").exists(), "tokenizer.model missing"
assert (REPO_DIR / "rag_index_dense" / "passages.jsonl").exists(), "rag index missing"
print(f"student.pt : {student.stat().st_size/1e9:.2f} GB")
print(f"All assets downloaded in {(time.time()-t0)/60:.1f} min")

In [ ]:
# ============================================================
# STEP 4 - Web UI + voice prompts fix
#   download_assets.sh / serve_s28_web.sh use allow_patterns voices/* and
#   dist/*, but the personaplex repo stores voices.tgz / dist.tgz at the
#   ROOT - those patterns match nothing, so $SNAP/dist would be missing
#   and aiohttp would crash at startup. Download + extract explicitly.
# ============================================================
import tarfile
from huggingface_hub import hf_hub_download, snapshot_download

snap = snapshot_download("nvidia/personaplex-7b-v1",
                         allow_patterns=["model.safetensors",
                                         "tokenizer-e351c8d8-checkpoint125.safetensors"],
                         token=HF_TOKEN)
print("personaplex snapshot:", snap)

for name in ("voices.tgz", "dist.tgz"):
    target = Path(snap) / name.replace(".tgz", "")
    if target.is_dir():
        print(f"{target.name}/ already present - skipping")
        continue
    arc = hf_hub_download("nvidia/personaplex-7b-v1", name, token=HF_TOKEN)
    with tarfile.open(arc, "r:gz") as tar:
        tar.extractall(path=Path(arc).parent)
    print(f"extracted {name} -> {target}")

assert (Path(snap) / "dist" / "index.html").exists(), "web UI (dist/index.html) still missing"
assert any((Path(snap) / "voices").iterdir()), "voices/ is empty"
print("Web UI + voice prompts OK")

In [ ]:
# ============================================================
# STEP 5 - Pre-download auxiliary models into the HF cache
#   (the server would download these on first start; doing it now makes
#    startup fast and the pod fully warm)
# ============================================================
from huggingface_hub import snapshot_download

snapshot_download("kyutai/stt-1b-en_fr-candle", token=HF_TOKEN)          # STT (~2.4 GB)
print("STT model cached")
snapshot_download("sentence-transformers/all-MiniLM-L6-v2")              # dense retriever
print("MiniLM cached")
snapshot_download("cross-encoder/ms-marco-MiniLM-L-6-v2")                # reranker
print("cross-encoder cached")
print("All auxiliary models cached in", HF_HOME)

In [ ]:
# ============================================================
# STEP 6 - Dry-run validation (loads everything, serves nothing)
# ============================================================
env = dict(os.environ, HF_TOKEN=HF_TOKEN, HF_HOME=str(HF_HOME), DRY="1")
rc = run_streamed("bash serve_s28_web.sh", cwd=REPO_DIR, env=env)
assert rc == 0, "dry run failed - check the log above"
print("\nDRY RUN OK: student + adapter + index + STT all load and step")

In [ ]:
# ============================================================
# STEP 7 - LAUNCH the streaming server on 0.0.0.0:8998
#   serve_s28_web.sh defaults: CKPT=ckpt/rag_S28/step_1800, HOST=0.0.0.0,
#   PORT=8998, RETRIEVER=hybrid (Tavily web + local index fallback)
# ============================================================
import time, urllib.request

# stop any previous instance (safe to re-run this cell to restart)
subprocess.run("pkill -f server_student_rag || true", shell=True)
time.sleep(2)

env = dict(os.environ, HF_TOKEN=HF_TOKEN, HF_HOME=str(HF_HOME))
# tip: for fully-offline serving (no web search) use the local index only:
# env["RETRIEVER"] = "local"
SERVER_LOG = open(LOG_FILE, "w")
SERVER_PROC = subprocess.Popen("bash serve_s28_web.sh", cwd=REPO_DIR, shell=True,
                               stdout=SERVER_LOG, stderr=subprocess.STDOUT,
                               env=env, start_new_session=True)
print(f"server PID: {SERVER_PROC.pid}   log: {LOG_FILE}")
print("waiting for the server to come up (first start loads ~14 GB of models)...")

ready = False
for _ in range(180):                       # up to ~15 min
    if SERVER_PROC.poll() is not None:     # process died -> show the log tail
        print("SERVER EXITED EARLY - last log lines:\n")
        print("".join(open(LOG_FILE).readlines()[-60:]))
        raise SystemExit("server failed to start")
    if LOG_FILE.exists() and "serving RAG student at" in open(LOG_FILE).read():
        ready = True
        break
    time.sleep(5)
assert ready, "server did not become ready within 15 min - check the log"

print("".join(open(LOG_FILE).readlines()[-12:]))
html = urllib.request.urlopen(f"http://127.0.0.1:{PORT}/", timeout=10).read()
print(f"\nHealth check: HTTP 200, {len(html)} bytes of web UI served")
print(f"STREAMING SERVER IS LIVE on port {PORT} - run the next cell for the URL")

In [ ]:
# ============================================================
# STEP 8 - Open the web UI
# ============================================================
pod_id = os.environ.get("RUNPOD_POD_ID", "").strip()
print("=" * 72)
print("OPEN THE STREAMING WEB UI:")
if pod_id:
    print(f"\n    https://{pod_id}-{PORT}.proxy.runpod.net\n")
else:
    print("\n  RunPod console -> your pod -> Connect -> 'HTTP Service [Port 8998]'")
    print(f"  (the URL looks like https://<pod-id>-{PORT}.proxy.runpod.net)\n")
print("Notes:")
print("  * Port 8998 must be exposed: Pod -> Edit -> Expose HTTP Service Ports.")
print("  * Use the HTTPS proxy URL - browsers block the microphone on plain http.")
print("  * The page streams duplex audio over a WebSocket to /api/chat.")
print("  * First turn may take a second while the retrieval warms up.")
print("=" * 72)

In [ ]:
# ============================================================
# Operations toolbox - run this cell once, then call the helpers anytime
# ============================================================

def tail_log(n=40):
    """Show the last n lines of the server log."""
    print("".join(open(LOG_FILE).readlines()[-n:]))

def server_status():
    """Is the streaming server running?"""
    r = subprocess.run("pgrep -af server_student_rag || echo 'NOT RUNNING'",
                       shell=True, capture_output=True, text=True)
    print(r.stdout)

def gpu():
    """nvidia-smi snapshot."""
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

def stop_server():
    """Stop the streaming server."""
    subprocess.run("pkill -f server_student_rag || true", shell=True)
    print("server stopped")

# to RESTART the server: just re-run the LAUNCH cell (STEP 7) - it kills the
# old instance first, then starts a fresh one.

server_status()
tail_log(15)

## Troubleshooting

| Symptom | Cause / fix |
|---|---|
| `401/403` on `nvidia/personaplex-7b-v1` | Gated repo: accept the license on its model page with the same account as your `HF_TOKEN`, then re-run from STEP 0 |
| GitHub clone has only a README | The code is not pushed yet — the notebook auto-fell-back to the HF zip. Push from Windows: `git init && git add -A && git commit -m "full code" && git branch -M main && git remote add origin https://github.com/MoshiHead/s_new_student_m_rag_v1.git && git push -u origin main --force` |
| Proxy URL does not load | Port **8998 not exposed**: Pod → Edit → *Expose HTTP Service Ports* → add `8998` |
| Mic does not work in the browser | Use the **https** proxy URL, not `http://<ip>:8998` |
| `torch.cuda.is_available()` False / no kernel image | Very old GPU (sm_75 or lower): delete `/workspace/menv` and rebuild with `TORCH_CUDA=cu121 bash setup_env.sh /workspace/menv` |
| pip builds `sphn` from source | Python 3.12+ has no sphn 0.1.12 wheel — recreate the env so it picks `python3.11` |
| OOM at startup | Less than ~16 GB free VRAM; close other processes (`nvidia-smi`), or set `env["RETRIEVER"] = "web"` in the launch cell to drop the local index models |
| Server died | `tail_log(60)` from the toolbox cell; re-run the launch cell to restart |
| Pod restarted | Everything persists on `/workspace` — just re-run the **Launch** cell (STEP 7) |

**Useful paths** (all on the persistent volume):
repo `/workspace/s_new_student_m_rag_v1` · env `/workspace/menv` · model cache `/workspace/.cache/huggingface` · server log `/workspace/s28_server.log`